# AgentMeter — FULL run on Colab L4 (5 models x 300 scenarios)

Runs `configs/run_full_l4.yaml` via `python main.py run-full`: all 5 models,
each in its own worker subprocess (sequential, one at a time), 4-bit NF4, over the
balanced 300-scenario CIC-IDS set. Results persist to SQLite **on Google Drive**
so they survive a session drop, and the run is **resumable**.

Runs top-to-bottom. A **free mock pre-flight** (Cell 8) validates the wiring on the
real 300-set with mock models before you pay for the L4 run (Cell 9).

> **Tahap 1 guardrails unchanged:** minimal linear pipeline (test subject only),
> sequential-only, Mode A, config-driven, 4-bit NF4 identical across all models.

## 1. GPU check
**L4** is required for the real run (Cell 9). CPU or T4 is fine for the free mock
pre-flight (Cell 8). Runtime → Change runtime type → **L4 GPU**.

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime -> Change runtime type -> L4 (real run) / any (mock pre-flight)'

## 2. Mount Google Drive
The SQLite DB is written under Drive so it survives a Colab disconnect.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/agentmeter_results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive results dir:', DRIVE_DIR)

## 3. Get the AgentMeter code
Clones the repo and **hard-resets** the working tree to `origin/BRANCH`, so a stale
clone from an earlier session can't linger. Private repo → paste a token (hidden);
public → press Enter.

In [ ]:
import os, getpass, subprocess
REPO = 'https://github.com/ismazahin/AgentMeter'
BRANCH = 'claude/cool-ride-mitmzl'
gh = getpass.getpass('GitHub token (press Enter if repo is public): ').strip()
url = REPO.replace('https://', f'https://{gh}@') if gh else REPO
if not os.path.isdir('/content/AgentMeter'):
    subprocess.run(['git', 'clone', url + '.git', '/content/AgentMeter'], check=True)
os.chdir('/content/AgentMeter')
subprocess.run(['git', 'remote', 'set-url', 'origin', url + '.git'], check=True)
subprocess.run(['git', 'fetch', '--force', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print('cwd   :', os.getcwd())
print('branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())
print('HEAD  :', subprocess.run(['git','log','-1','--oneline'],capture_output=True,text=True).stdout.strip())

## 4. Install dependencies
Colab already ships a CUDA `torch`; we don't reinstall it. Add the CPU deps plus
the GPU/HF extras and `bitsandbytes` for 4-bit.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers accelerate huggingface_hub sentencepiece pynvml bitsandbytes

## 5. Hugging Face token
Entered via `getpass` or a Colab Secret named `HF_TOKEN` — **not** hardcoded, not
printed. **Accept the gated licences first** on each model's HF page:
`mistralai/Mistral-7B-Instruct-v0.3`, `meta-llama/Meta-Llama-3-8B-Instruct`, and
`google/gemma-2-9b-it` (Qwen and Phi-3 are ungated). Otherwise the download 401s.

In [ ]:
import os, getpass
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not tok:
    tok = getpass.getpass('Enter your HF_TOKEN (hidden): ').strip()
assert tok, 'HF_TOKEN is required for the gated models.'
os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set (', len(tok), 'chars ). Not displayed.')

## 6. Upload the 300-scenario dataset
Upload `cicids_full_300.csv` (produced by `python main.py build-dataset` on your
machine). It's moved to `data/cicids_full_300.csv` and validated: shape (300, 79)
and 60 rows per class — the cell **fails loudly** if not.

In [ ]:
import os, shutil, pandas as pd
from google.colab import files

os.makedirs('data', exist_ok=True)
uploaded = files.upload()   # pick cicids_full_300.csv
csvs = [f for f in uploaded if f.lower().endswith('.csv')]
assert csvs, 'No .csv uploaded — choose cicids_full_300.csv.'
src = csvs[0]
dst = 'data/cicids_full_300.csv'
if os.path.abspath(src) != os.path.abspath(dst):
    shutil.move(src, dst)

df = pd.read_csv(dst)
dist = df['label'].value_counts().to_dict()
print('shape   :', df.shape)
print('classes :', dist)
EXPECT = {'Brute Force', 'Volumetric DDoS', 'Port Scanning', 'DoS Hulk', 'Benign'}
assert df.shape == (300, 79), f'FAIL: expected (300, 79), got {df.shape}'
assert set(dist) == EXPECT, f'FAIL: classes {set(dist)} != {EXPECT}'
assert all(v == 60 for v in dist.values()), f'FAIL: not 60/class -> {dist}'
print('OK: 300 rows x 79 cols, 60 per class.')

## 7. Point results/ at Drive
Symlink `results/` to the Drive folder so the SQLite DB (and everything under
`results/`) is written straight to Drive and survives a session drop.

In [ ]:
import os, subprocess
subprocess.run(['rm', '-rf', 'results'])
os.symlink(DRIVE_DIR, 'results')
print('results ->', os.path.realpath('results'))

## 8. FREE pre-flight (mock models, real 300-set)
Runs the **full** `run-full` path over the real 300 scenarios with **mock** models
— no GPU, no HF, no cost — to catch any config/wiring error before you pay. Expect
1 run row and, per mock model, 300 `scenario_results` + 1200 `agent_metrics`.

In [ ]:
!python main.py --config configs/run_full_mock300.yaml run-full --fresh

import sqlite3
db = 'results/agentmeter_mock300.db'
c = sqlite3.connect(db); c.row_factory = sqlite3.Row
print('\nSQLite row counts (mock pre-flight):')
for t in ('runs', 'scenario_results', 'agent_metrics'):
    print(f'  {t:<17}', c.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])
print('per-model scenario_results:')
for r in c.execute('SELECT model, COUNT(*) n FROM scenario_results GROUP BY model ORDER BY model'):
    print('   ', r['model'], r['n'])
c.close()

## 9. REAL RUN — the paid L4 step 💸
Loads the 5 real models in 4-bit on the L4, one worker per model. The cell **stops**
unless it's running on an L4 (VRAM/latency are only comparable on one GPU). Each
model's HF cache is freed after it finishes (`cleanup_model_cache_after: true`) so
the disk won't fill.

**Resumable:** if the session disconnects, just re-run **this same cell** (no
`--fresh`) — it skips completed (model, scenario) pairs and continues.

In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
if 'L4' not in gpu:
    print('!' * 70)
    print(f'WARNING: GPU is {gpu!r}, not an L4.')
    print('VRAM/latency are only comparable on ONE GPU model. Switch Runtime ->')
    print('Change runtime type -> L4 before the real run, or the numbers will not match.')
    print('!' * 70)
assert 'L4' in gpu, f'Refusing the paid run on a non-L4 GPU: {gpu}'

# Resumable: re-run this cell after a disconnect to continue (no --fresh).
!python main.py --config configs/run_full_l4.yaml run-full

## 10. Verify + results
Final SQLite row counts per model, read from the DB on Drive. Expect per model:
300 `scenario_results` and 1200 `agent_metrics` (4 agents x 300). The DB lives at
`/content/drive/MyDrive/agentmeter_results/` — download it or keep it on Drive.

**When done: Runtime → Disconnect and delete runtime to stop L4 billing.**

In [ ]:
import sqlite3
db = 'results/agentmeter_full_l4.db'
c = sqlite3.connect(db); c.row_factory = sqlite3.Row
print('runs:', c.execute('SELECT COUNT(*) FROM runs').fetchone()[0])
print('per-model scenario_results:')
for r in c.execute('SELECT model, COUNT(*) n FROM scenario_results GROUP BY model ORDER BY model'):
    print('   ', f"{r['model']:<40}", r['n'])
print('per-model agent_metrics:')
for r in c.execute('SELECT model, COUNT(*) n FROM agent_metrics GROUP BY model ORDER BY model'):
    print('   ', f"{r['model']:<40}", r['n'])
c.close()
print('\nDB persisted on Drive:', DRIVE_DIR)
print('Now: Runtime -> Disconnect and delete runtime to stop billing.')